# Mathematica-Style Equilibrium Panel

Produces a 2×2 composite panel for the Baseline model, matching the visual
style of `graphtest-money.nb` (purple/orange colour scheme, percentage tick
labels, Mathematica-style layout).

**Requires:** `PKAssetPrices`, `CairoMakie`

In [ ]:
using PKAssetPrices
using PKAssetPrices.Static: solve_model, eval_curve
using CairoMakie
using Printf

# ── Colour palette (Mathematica graphtest-money.nb) ──
const CURVE_COLOR  = RGBf(0.5, 0.0, 0.5)    # purple  – IS, AD, asset demand
const SUPPLY_COLOR = RGBf(1.0, 0.5, 0.0)    # orange  – IR, AS, asset supply
const COUNTERFACT  = (CURVE_COLOR, 0.38)     # faded purple – counterfactual

# Balance-sheet palette
const BS_ASSET     = RGBf(0.20, 0.59, 0.86)  # blue   – assets
const BS_LIABILITY = RGBf(0.84, 0.25, 0.30)  # red    – liabilities
const BS_ASSET_LT  = RGBf(0.35, 0.72, 0.93)  # light blue  – reserves as asset
const BS_LIAB_LT   = RGBf(0.92, 0.50, 0.55)  # light red   – CB credit as liab

const PLOT_RANGES = (
    IS_IR_X = (4.0, 10.0),
    IS_IR_Y = (0.06, 0.15),
    AD_AS_X = (5.0, 9.0),
    AD_AS_Y = (0.9, 2.5),
    AM_X    = (0.6, 1.3),
    AM_Y    = (0.7, 1.5),
)

FONT_SIZE   = 20
TITLE_SIZE  = 26
LEGEND_SIZE = 20

In [ ]:
# ── 1. Solve model ─────────────────────────────────────────────────────
solution = solve_model(PKAssetPrices.Static.Baseline)
vars     = solution.variables

Y_eq   = vars[:Y]
r_eq   = vars[:r]
P_eq   = vars[:P]
AP_eq  = vars[:AP]
dL_val = vars[:dL]
dM_val = vars[:dM]
dR_val = vars[:dR]

println("Baseline model solved.")
println("  Y  = $(round(Y_eq, sigdigits=5))")
println("  r  = $(round(r_eq, sigdigits=5))")
println("  P  = $(round(P_eq, sigdigits=5))")
println("  AP = $(round(AP_eq, sigdigits=5))")

In [ ]:
# ── 2. Build figure ─────────────────────────────────────────────────────
fig = Figure(
    size = (1400, 1000),
    fontsize = FONT_SIZE,
    figure_padding = (28, 38, 28, 28),
    backgroundcolor = :white,
)

# ── Panel A: Goods Market Dynamics (IS/IR) ───────────────────────────
ax1 = Axis(
    fig[1, 1];
    title = "(A) Goods Market Dynamics",
    xlabel = "Output Y",
    ylabel = "interest rate r",
    xgridcolor = (:black, 0.12),
    ygridcolor = (:black, 0.12),
    titlesize = TITLE_SIZE,
    xlabelsize = FONT_SIZE,
    ylabelsize = FONT_SIZE,
    bottomspinecolor = :black,
    topspinecolor = :black,
    leftspinecolor = :black,
    rightspinecolor = :black,
)

# IS curve: r → Y
r_range = range(PLOT_RANGES.IS_IR_Y...; length = 200)
is_Y = map(r_range) do r
    v = copy(vars); v[:r] = r
    eval_curve(solution.model, v).IS
end
lines!(ax1, is_Y, r_range; color = CURVE_COLOR, linewidth = 3, label = "IS")

# IR curve: Y → r
y_range1 = range(PLOT_RANGES.IS_IR_X...; length = 200)
ir_r = map(y_range1) do y
    v = copy(vars); v[:Y] = y
    eval_curve(solution.model, v).IR
end
lines!(ax1, y_range1, ir_r; color = SUPPLY_COLOR, linewidth = 3, label = "IR")

# Counterfactual IR (lower i₀)
lower_model = PKAssetPrices.Static.Parametrization(
    solution.model.model,
    merge(copy(solution.model.params), Dict(:i0 => 0.0)),
    copy(solution.model.u0),
)
lower_ir_r = map(y_range1) do y
    v = copy(vars); v[:Y] = y
    eval_curve(lower_model, v).IR
end
lines!(ax1, y_range1, lower_ir_r;
    color = COUNTERFACT, linewidth = 2, linestyle = :dash,
    label = "IR (lower i₀)")

xlims!(ax1, PLOT_RANGES.IS_IR_X...)
ylims!(ax1, PLOT_RANGES.IS_IR_Y...)
axislegend(ax1; position = :rb, framevisible = false, labelsize = LEGEND_SIZE)
ax1

In [ ]:
# ── Panel B: Output and Inflation Dynamics (AD/AS) ──────────────────
ax2 = Axis(
    fig[1, 2];
    title = "(B) Output and Inflation Dynamics",
    xlabel = "Output Y",
    ylabel = "Price Level P",
    xgridcolor = (:black, 0.12),
    ygridcolor = (:black, 0.12),
    titlesize = TITLE_SIZE,
    xlabelsize = FONT_SIZE,
    ylabelsize = FONT_SIZE,
    bottomspinecolor = :black,
    topspinecolor = :black,
    leftspinecolor = :black,
    rightspinecolor = :black,
)

# AD curve: P → Y
p_range = range(PLOT_RANGES.AD_AS_Y...; length = 200)
ad_Y = map(p_range) do p
    v = copy(vars); v[:P] = p
    eval_curve(solution.model, v).ADc
end
lines!(ax2, ad_Y, p_range; color = CURVE_COLOR, linewidth = 3, label = "AD")

# AS curve: Y → P
y_range2 = range(PLOT_RANGES.AD_AS_X...; length = 200)
as_P = map(y_range2) do y
    v = copy(vars); v[:Y] = y
    eval_curve(solution.model, v).ASc
end
lines!(ax2, y_range2, as_P; color = SUPPLY_COLOR, linewidth = 3, label = "AS")

# AD counterfactual (lower i₀)
lower_ad_Y = map(p_range) do p
    v = copy(vars); v[:P] = p
    eval_curve(lower_model, v).ADc
end
lines!(ax2, lower_ad_Y, p_range;
    color = (CURVE_COLOR, 0.38), linewidth = 2, linestyle = :dash,
    label = "AD (lower i₀)")

# Equilibrium point
vlines!(ax2, [Y_eq]; color = (:black, 0.20), linestyle = :dot, linewidth = 1.5)
hlines!(ax2, [P_eq]; color = (:black, 0.20), linestyle = :dot, linewidth = 1.5)
scatter!(ax2, [Y_eq], [P_eq]; color = :black, markersize = 12,
    strokecolor = :white, strokewidth = 2)

xlims!(ax2, PLOT_RANGES.AD_AS_X...)
ylims!(ax2, PLOT_RANGES.AD_AS_Y...)
axislegend(ax2; position = :rb, framevisible = false, labelsize = LEGEND_SIZE)
ax2

In [ ]:
# ── Panel C: Financial Market Dynamics (Asset Market) ──────────────
ax3 = Axis(
    fig[2, 1];
    title = "(C) Financial Market Dynamics",
    xlabel = "Base-price-equivalent quantity",
    ylabel = "Asset Price AP",
    xgridcolor = (:black, 0.12),
    ygridcolor = (:black, 0.12),
    titlesize = TITLE_SIZE,
    xlabelsize = FONT_SIZE,
    ylabelsize = FONT_SIZE,
    bottomspinecolor = :black,
    topspinecolor = :black,
    leftspinecolor = :black,
    rightspinecolor = :black,
)

ap_range = range(PLOT_RANGES.AM_X...; length = 200)

# Asset demand (AMD)
amd_Q = map(ap_range) do ap
    v = copy(vars); v[:AP] = ap
    eval_curve(solution.model, v).AMD
end
lines!(ax3, amd_Q, ap_range; color = CURVE_COLOR, linewidth = 3,
    label = "Asset Demand")

# Asset supply (AMS)
ams_Q = map(ap_range) do ap
    v = copy(vars); v[:AP] = ap
    eval_curve(solution.model, v).AMS
end
lines!(ax3, ams_Q, ap_range; color = SUPPLY_COLOR, linewidth = 3,
    label = "Asset Supply")

# Counterfactual demand (lower i₀)
lower_sol = solve_model(lower_model)
lower_amd_Q = map(ap_range) do ap
    v = copy(lower_sol.variables); v[:AP] = ap
    eval_curve(lower_model, v).AMD
end
lines!(ax3, lower_amd_Q, ap_range;
    color = (CURVE_COLOR, 0.38), linewidth = 2, linestyle = :dash,
    label = "Demand (lower i₀)")

# Equilibrium point
curves_eq = eval_curve(solution)
Q_eq = (curves_eq.AMD + curves_eq.AMS) / 2
vlines!(ax3, [Q_eq]; color = (:black, 0.20), linestyle = :dot, linewidth = 1.5)
hlines!(ax3, [AP_eq]; color = (:black, 0.20), linestyle = :dot, linewidth = 1.5)
scatter!(ax3, [Q_eq], [AP_eq]; color = :black, markersize = 12,
    strokecolor = :white, strokewidth = 2)

xlims!(ax3, PLOT_RANGES.AM_X...)
ylims!(ax3, PLOT_RANGES.AM_Y...)
axislegend(ax3; position = :rt, framevisible = false, labelsize = LEGEND_SIZE)
ax3

In [ ]:
# ── Panel D: Sector Balance Sheets ───────────────────────────────────
ax4 = Axis(
    fig[2, 2];
    title = "Sector Balance Sheets",
    ylabel = "Amount",
    xgridvisible = false,
    ygridcolor = (:black, 0.08),
    titlesize = TITLE_SIZE,
    xlabelsize = FONT_SIZE,
    ylabelsize = FONT_SIZE,
    bottomspinecolor = :black,
    topspinecolor = :black,
    leftspinecolor = :black,
    rightspinecolor = :black,
)

# 8 segments, 6 bar groups (stacking)
positions  = [1.0, 2.0, 3.0, 3.0, 4.0, 4.0, 5.0, 6.0]
bar_heights = [dM_val, dL_val, dL_val, dR_val, dM_val, dR_val, dR_val, dR_val]
bar_groups  = Int[1, 2, 3, 3, 4, 4, 5, 6]
bar_colors  = [
    BS_ASSET,       # 1: PS Assets – deposits
    BS_LIABILITY,   # 2: PS Liabilities – loans
    BS_ASSET,       # 3a: Bank Assets – loans
    BS_ASSET_LT,    # 3b: Bank Assets – reserves
    BS_LIABILITY,   # 4a: Bank Liabilities – deposits
    BS_LIAB_LT,     # 4b: Bank Liabilities – CB credit
    BS_ASSET,       # 5: CB Assets – CB credit
    BS_LIABILITY,   # 6: CB Liabilities – reserves
]

barplot!(ax4, positions, bar_heights;
    stack = bar_groups, width = 0.82, color = bar_colors,
    strokecolor = (:black, 0.65), strokewidth = 1.5,
    bar_labels = [
        "Deposits\n$(round(dM_val; sigdigits=4))",
        "Loans\n$(round(dL_val; sigdigits=4))",
        "Loans\n$(round(dL_val; sigdigits=4))",
        "Reserves\n$(round(dR_val; sigdigits=4))",
        "Deposits\n$(round(dM_val; sigdigits=4))",
        "CB Credit\n$(round(dR_val; sigdigits=4))",
        "CB Credit\n$(round(dR_val; sigdigits=4))",
        "Reserves\n$(round(dR_val; sigdigits=4))",
    ],
    label_position = :center, label_color = :black, label_size = 14,
)

hlines!(ax4, [0.0]; color = (:black, 0.55), linewidth = 1.5)

# Sector labels
max_y = max(dM_val, dL_val + dR_val) * 1.18
text!(ax4, [1.5, 3.5, 5.5], fill(max_y * 0.90, 3);
    text = ["Private Sector", "Banks", "Central Bank"],
    align = (:center, :bottom), font = :bold, fontsize = TITLE_SIZE,
    color = :black)

# Annotation
rr   = dR_val / dM_val
risk = get(vars, :SD, 0.0) / dL_val
annotation = join([
    @sprintf("Reserve ratio: %.4f", rr),
    @sprintf("Total loans: %.4f", dL_val),
    @sprintf("Risk indicator: %.4f", risk),
], '\n')
text!(ax4, 0.98, 0.50; text = annotation, space = :relative,
    align = (:right, :center), fontsize = FONT_SIZE, color = :black)

tick_labels = [
    "Assets\nPrivate\nSector",
    "Liabilities\nPrivate\nSector",
    "Assets\nBanks",
    "Liabilities\nBanks",
    "Assets\nCentral\nBank",
    "Liabilities\nCentral\nBank",
]
ax4.xticks = (1.0:1.0:6.0, tick_labels)
xlims!(ax4, 0.3, 6.7)
ylims!(ax4, 0.0, max_y)
ax4

In [ ]:
# ── 3. Layout & display ────────────────────────────────────────────────
colgap!(fig.layout, 42)
rowgap!(fig.layout, 22)
rowsize!(fig.layout, 1, Relative(0.48))
rowsize!(fig.layout, 2, Relative(0.52))

fig  # display inline

In [ ]:
# ── 4. Export ───────────────────────────────────────────────────────────
output_dir = normpath(joinpath(@__DIR__, "..", "..", "plots"))
mkpath(output_dir)

pdf_path = joinpath(output_dir, "baseline_mathematica_style.pdf")
png_path = joinpath(output_dir, "baseline_mathematica_style.png")

save(pdf_path, fig; pt_per_unit = 2)
println("Saved: $pdf_path")
save(png_path, fig)
println("Saved: $png_path")

In [ ]:
# ── 5. Verify equilibrium values ────────────────────────────────────────
println("\n── Equilibrium values ──")
for (k, v) in sort!(collect(vars); by = first)
    println(@sprintf("  %12s = %.6f", k, v))
end